In [ ]:
import sys
import os
from pathlib import Path
import wfdb
import pandas as pd
import numpy as np

# Resolve repo root regardless of execution directory
REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))
from src.features.hrv import get_window_features

# ── CONFIG ────────────────────────────────────────────────────────────────
# NOTE: PATIENTS must match the list in Notebook 02 exactly.
# When infants 2-9 finish downloading: update BOTH notebooks.
PATIENTS      = ["infant1", "infant10"]
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
RAW_DIR       = REPO_ROOT / "data" / "raw" / "physionet.org" / "files" / "picsdb" / "1.0.0"
FS_ECG        = 500    # Hz
WINDOW_SIZE   = 50     # beats
STEP_SIZE     = 25     # beats — 50% overlap

print(f"REPO_ROOT:     {REPO_ROOT}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"RAW_DIR:       {RAW_DIR}")

In [ ]:
def extract_features(patient_id):
    """
    Sliding window HRV feature extraction for one patient.
    Output columns (locked to get_window_features() return keys):
      record_name, window_idx, rr_ms_mean, rr_ms_std, rr_ms_min,
      rr_ms_max, rr_ms_25%, rr_ms_50%, rr_ms_75%
    """
    rr_path = PROCESSED_DIR / f"{patient_id}_rr_clean.csv"
    rr_ms   = pd.read_csv(rr_path)["rr_ms"].values

    rows      = []
    win_idx   = 0
    start     = 0

    while start + WINDOW_SIZE <= len(rr_ms):
        window = rr_ms[start : start + WINDOW_SIZE]
        row    = get_window_features(window, patient_id, win_idx)
        rows.append(row)
        start   += STEP_SIZE
        win_idx += 1

    df = pd.DataFrame(rows)

    # Enforce exact column order — crash loudly if schema drifted
    expected_cols = [
        "record_name", "window_idx",
        "rr_ms_mean", "rr_ms_std", "rr_ms_min",
        "rr_ms_max", "rr_ms_25%", "rr_ms_50%", "rr_ms_75%"
    ]
    missing = [c for c in expected_cols if c not in df.columns]
    assert not missing, f"Missing columns from get_window_features(): {missing}"

    return df[expected_cols]

In [ ]:
def extract_labels(patient_id):
    """
    Load bradycardia annotations from .atr file.
    Saves sample_idx and symbol only.
    Beat-to-window alignment is deferred to Notebook 04,
    which will use cumulative RR sum to map sample_idx → window_idx.
    """
    ann_path = RAW_DIR / f"{patient_id}_ecg"
    ann      = wfdb.rdann(str(ann_path), 'atr')

    rows = [
        {"sample_idx": int(s), "symbol": sym}
        for s, sym in zip(ann.sample, ann.symbol)
    ]

    df = pd.DataFrame(rows)
    print(f"  {patient_id}: {len(df)} annotations")
    print(f"  symbols: {sorted(df['symbol'].unique())}")
    return df

In [ ]:
for patient_id in PATIENTS:
    print(f"\n── {patient_id} ──────────────────────────────")

    # Features
    features_df = extract_features(patient_id)
    feat_path   = PROCESSED_DIR / f"{patient_id}_features.csv"
    features_df.to_csv(feat_path, index=False)
    print(f"  features: {features_df.shape} → {feat_path}")

    # Labels
    labels_df  = extract_labels(patient_id)
    label_path = PROCESSED_DIR / f"{patient_id}_labels.csv"
    labels_df.to_csv(label_path, index=False)
    print(f"  labels:   {labels_df.shape} → {label_path}")

print("\n✅ Notebook 03 complete.")